# CIFAR-10 frozen-visual replication — final 8-seed version

Shared definitions are followed by the final CIFAR-10 block from `v4-results(3).ipynb`. Boundary/sufficiency measurements use the auxiliary seed set; headline inversion and four-arm runs use eight seeds. The superseded earlier CIFAR cell was removed.


In [ ]:
# !pip install torch torchvision numpy pandas matplotlib

import copy, glob, hashlib, itertools, json, math, os, pathlib, platform, random, time
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
MODALITY = "cifar10"        # "abstract" | "mnist" | "cifar10" | "cifar100"
TARGET   = "encoder"         # visual only: "encoder" | "true_label"
PRESET   = "full"            # "smoke" | "quick" | "full"
RESUME   = True              # skip any unit of work already in results/
WORKDIR  = os.environ.get("PAPER_WORKDIR", ".")   # point at Drive to survive Colab
# =========================================================================
#
# PRESET = "smoke" runs every block on the full grid with tiny budgets. It is
# a correctness check, not a result: ~15 minutes on a GPU, and it proves the
# whole pipeline executes before you commit a day of compute to "full".
#
# RESUME = True means a killed session loses at most one run. Every finished
# run is appended to results/<block>__<modality>.jsonl and skipped on restart.
# Delete a block's .jsonl to force it to recompute.

RUN = {"sanity": True, "free": True, "boundary": True, "costforms": True,
       "width": True, "ood": True, "replication": True, "cores": True,
       "schedule": True, "inversion": True, "switch": True, "silencing": True,
       "reversibility": True, "negatives": True, "lamstar": True, "swap": True,
       "sharebias": True, "figures": True, "derived": True, "appendix": True,
       "mamba": True}

IS_VISUAL = MODALITY != "abstract"
TAG = MODALITY

WORK = pathlib.Path(WORKDIR).expanduser().resolve()
# smoke and quick outputs live in their own tree, so a pipeline check can never
# be mixed into the paper's numbers by a resumed sweep picking up stale records
_sfx = "" if PRESET == "full" else f"_{PRESET}"
RESDIR, CKPTDIR = WORK / f"results{_sfx}", WORK / f"checkpoints{_sfx}"
FIGDIR, TABDIR = WORK / f"figures{_sfx}", WORK / f"tables{_sfx}"
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)


def S(full, quick=None, smoke=None):
    """Budget for the current preset. Defaults keep the ratios sane."""
    return {"full": full,
            "quick": quick if quick is not None else max(full // 3, 100),
            "smoke": smoke if smoke is not None else max(full // 40, 100)}[PRESET]


# Headline results (boundary, inversion, OOD, switch, reversibility, width) get
# five seeds; the large factorial grids get three. Both drop to one under smoke.
SEEDS_MAIN = {"full": [0, 1, 2, 3, 4], "quick": [0, 1], "smoke": [0]}[PRESET]
SEEDS_AUX  = {"full": [0, 1, 2],       "quick": [0, 1], "smoke": [0]}[PRESET]

# One accuracy floor for every crossing in the paper. A crossing is only
# meaningful where the model still solves the task, and the floor decides which
# cells are excluded — so it has to be the same number everywhere or the
# crossings are not comparable across sections. State this value in the paper.
ACC_FLOOR = 0.90

STEPS = S(2500, 1200, 150)     # default per-run budget
LONG  = S(16000, 4000, 400)    # drift / equilibrium budget
XLONG = S(24000, 6000, 600)    # the long-budget arm of the Delta measurement (S6.1)

print(f"device {DEVICE} | modality {MODALITY} | preset {PRESET} | resume {RESUME}")
print(f"work dir {WORK}")
print(f"seeds: main {SEEDS_MAIN}, aux {SEEDS_AUX} | steps {STEPS} / {LONG} / {XLONG}")
if DEVICE.type == "cpu":
    print("!! CPU — roughly 10x slower. Colab: Runtime -> Change runtime type -> GPU.")
if PRESET == "smoke":
    print("!! SMOKE preset: numbers are meaningless. Use it to check the pipeline runs.")


def csv(name):
    return str(RESDIR / f"{name}__{TAG}.csv")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


## Persistence and resume

A full run is several million optimiser steps, which is longer than a Colab
session. Every unit of work is content-hashed on its specification and appended
to a JSONL file the moment it finishes, so restarting the notebook re-reads what
is already done and runs only what is missing.

Two granularities. Sweeps resume at the level of one `(condition, seed)` run.
Runs of 8k steps and up also checkpoint model and optimiser state mid-flight, so
a 24,000-step run survives a disconnect. A resumed run restarts its data stream
from a fresh RNG state: since trials are generated i.i.d. online this is
statistically equivalent, but it is not bit-identical, and initialisation is
still exactly seeded either way.

To force a recomputation, delete that block's `.jsonl`.


In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, pathlib.Path):
        return str(o)
    return str(o)


def keyof(spec):
    """Content hash of a run specification. Two runs with the same spec are the
    same experiment, so the second one is skipped."""
    return hashlib.md5(
        json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


def cfg_tag(cfg=None):
    if cfg is None:
        return "default"
    return ",".join(f"{k}={getattr(cfg, k)}" for k in sorted(cfg.__dataclass_fields__))


class Store:
    """Append-only JSONL of finished units of work, one file per block.

    A unit is whatever a block can afford to lose: usually one (condition, seed)
    training run. Records are flushed immediately, so a killed kernel loses at
    most the run in flight. A truncated final line is discarded on load.
    """

    def __init__(self, block):
        self.block = block
        self.path = RESDIR / f"{block}__{TAG}.jsonl"
        self.recs, bad = {}, 0
        if RESUME and self.path.exists():
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                    self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] discarded {bad} unreadable record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n")
            f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        """Flatten to one row per result. A record either carries metric fields
        directly or a `rows` list, in which case each entry becomes a row."""
        out = []
        for r in self.recs.values():
            spec = {k: v for k, v in r["_spec"].items()}
            body = {k: v for k, v in r.items()
                    if not k.startswith("_") and k not in ("trace", "rows")}
            if "rows" in r:
                for row in r["rows"]:
                    out.append({**spec, **body, **row})
            else:
                out.append({**spec, **body})
        return pd.DataFrame(out)

    def traces(self):
        out = []
        for r in self.recs.values():
            for t in r.get("trace", []):
                out.append({**r["_spec"], **t})
        return pd.DataFrame(out)

    def elapsed(self):
        return sum(r.get("_elapsed", 0.0) for r in self.recs.values())


def cached(store, spec, fn, label=None):
    """Run `fn()` unless this exact spec is already on disk. `fn` returns the
    payload dict; anything not JSON-serialisable must be checkpointed separately."""
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if label:
        print(f"  [{store.block}] {label}  {payload['_elapsed']/60:.1f}m")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}__{TAG}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    torch.save(obj, str(p) + ".tmp")
    os.replace(str(p) + ".tmp", str(p))


def ckpt_load(name):
    p = ckpt_path(name)
    if not (RESUME and p.exists()):
        return None
    try:
        return torch.load(str(p), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(p), map_location=DEVICE)
    except Exception as e:
        print(f"  checkpoint {p.name} unreadable ({e}) — recomputing")
        return None


def save_model(name, m):
    ckpt_save(name, {"state": m.state_dict()})


def load_model_into(name, m):
    ck = ckpt_load(name)
    if ck is None:
        return False
    m.load_state_dict(ck["state"])
    return True


print(f"persistence: results -> {RESDIR}   checkpoints -> {CKPTDIR}")


## Configuration

`n_out` is the readout width and therefore chance level: 8 for symbols, 10 for
MNIST/CIFAR-10, 100 for CIFAR-100. Dependency scores are normalised by each
model's own headroom above its own chance, so the measures are comparable
across modalities without rescaling.

In [ ]:
PAD, DELAY, CUE, MARK = 0, 1, 2, 3
N_SPECIAL = 4
MAX_POS = 24
D_FEAT = 64


@dataclass
class Config:
    n_symbols: int = 8            # abstract alphabet
    max_query_pos: int = MAX_POS
    base_delay: int = 3
    d_model: int = 64
    lr: float = 3e-3
    batch_size: int = 128
    weight_decay: float = 0.0
    grad_clip: float = 1.0


CFG = Config()
N_CLASSES = {"abstract": CFG.n_symbols, "mnist": 10, "cifar10": 10,
             "cifar100": 100}[MODALITY]
# visual packing: [features | query one-hot | is_delay | is_query | is_mark]
IN_DIM = D_FEAT + MAX_POS + 3


def vocab_size(cfg):
    return N_SPECIAL + cfg.n_symbols + cfg.max_query_pos


def token_str(t, cfg=CFG):
    t = int(t)
    return {PAD: "_", DELAY: ".", CUE: "?", MARK: "*"}.get(
        t, chr(ord("A") + t - N_SPECIAL) if t < N_SPECIAL + cfg.n_symbols
        else f"Q{t - N_SPECIAL - cfg.n_symbols}")

## Frozen encoder (visual modalities only)

`AdaptiveAvgPool2d(3)` rather than `(1)`: collapsing the feature map to one
number per channel discards the spatial structure digits and objects depend on,
and cost about seven points of MNIST accuracy in an earlier version.

The encoder is **frozen** after training. A jointly-trained encoder would be a
third pathway able to compress or preserve information, and the two-route
decomposition every measurement here relies on would no longer partition the
information flow.

In [ ]:
BANK = {}

if IS_VISUAL:
    import torchvision
    from torchvision import transforms as T

    class Encoder(nn.Module):
        def __init__(self, in_ch, width, d_feat=D_FEAT, n_classes=10):
            super().__init__()
            a, b, c = width
            self.conv = nn.Sequential(
                nn.Conv2d(in_ch, a, 3, padding=1), nn.BatchNorm2d(a), nn.ReLU(),
                nn.Conv2d(a, a, 3, padding=1), nn.BatchNorm2d(a), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(),
                nn.Conv2d(b, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(b, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(),
                nn.AdaptiveAvgPool2d(3))
            self.fc = nn.Linear(c * 9, d_feat)
            self.head = nn.Linear(d_feat, n_classes)

        def features(self, x):
            return self.fc(self.conv(x).flatten(1))

        def forward(self, x):
            return self.head(F.relu(self.features(x)))

    SPEC = {
        "mnist":    dict(cls=torchvision.datasets.MNIST, ch=1, ep=4,
                         w=(32, 64, 128), mean=(0.1307,), std=(0.3081,), aug=False),
        "cifar10":  dict(cls=torchvision.datasets.CIFAR10, ch=3, ep=20,
                         w=(64, 128, 256), mean=(0.4914, 0.4822, 0.4465),
                         std=(0.247, 0.243, 0.261), aug=True),
        "cifar100": dict(cls=torchvision.datasets.CIFAR100, ch=3, ep=30,
                         w=(64, 128, 256), mean=(0.5071, 0.4865, 0.4409),
                         std=(0.2673, 0.2564, 0.2762), aug=True),
    }[MODALITY]

    base = [T.ToTensor(), T.Normalize(SPEC["mean"], SPEC["std"])]
    tf_tr = T.Compose(([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()]
                       if SPEC["aug"] else []) + base)
    tf_te = T.Compose(base)
    ds_tr = SPEC["cls"]("./data", train=True, download=True, transform=tf_tr)
    ds_te = SPEC["cls"]("./data", train=False, download=True, transform=tf_te)

    set_seed(0)
    enc = Encoder(SPEC["ch"], SPEC["w"], n_classes=N_CLASSES).to(DEVICE)
    opt_e = torch.optim.AdamW(enc.parameters(), lr=2e-3, weight_decay=5e-4)
    dl = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True,
                                     num_workers=2, drop_last=True)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt_e, 2e-3, SPEC["ep"] * len(dl))
    for ep in range(SPEC["ep"]):
        enc.train()
        for xb, yb in dl:
            loss = F.cross_entropy(enc(xb.to(DEVICE)), yb.to(DEVICE))
            opt_e.zero_grad(set_to_none=True); loss.backward(); opt_e.step(); sch.step()
        if (ep + 1) % max(SPEC["ep"] // 5, 1) == 0:
            print(f"  encoder epoch {ep+1}/{SPEC['ep']}  loss {loss.item():.3f}")
    enc.eval()
    for p in enc.parameters():
        p.requires_grad_(False)

    @torch.no_grad()
    def encode_all(dataset, perturb=None, bs=512):
        dl = torch.utils.data.DataLoader(dataset, batch_size=bs, num_workers=2)
        fs, ys, ps = [], [], []
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            if perturb is not None:
                xb = perturb(xb)
            f = enc.features(xb)
            fs.append(f); ys.append(yb.to(DEVICE))
            ps.append(enc.head(F.relu(f)).argmax(-1))
        return torch.cat(fs), torch.cat(ys), torch.cat(ps)

    f_tr, y_tr, p_tr = encode_all(ds_tr)
    f_te, y_te, p_te = encode_all(ds_te)
    ENC_ACC = (p_te == y_te).float().mean().item()
    BANK = {"f_tr": f_tr, "t_tr": p_tr if TARGET == "encoder" else y_tr,
            "f_te": f_te, "t_te": p_te if TARGET == "encoder" else y_te,
            "y_te": y_te, "p_te": p_te, "enc_acc": ENC_ACC, "test_ds": ds_te}
    print(f"\nfrozen encoder test accuracy: {ENC_ACC:.4f}  ({N_CLASSES} classes, "
          f"feature dim {D_FEAT})")
    print(f"target = {TARGET}  →  working-memory ceiling is "
          f"{'1.000 regardless of encoder quality' if TARGET=='encoder' else f'{ENC_ACC:.3f}'}")

## Tasks

Four paradigms, defined once and dispatched on modality. `selective_recall` is
primary: the query is unknown until the end, so nothing can be stored
selectively. `distractor` holds the retention interval fixed and varies
interference alone, dissociating two things that grow together elsewhere.

`affine_track` is the **positive control**. It composes maps `s ↦ (a·s+b) mod 8`
with `a` coprime to 8; composition does not commute, so a permutation-invariant
attention read provably cannot succeed and recurrence must be used. Before any
`hidden_dep = 0` elsewhere can be interpreted, the measure has to be shown
capable of reporting recurrence. In visual modalities each image's class
indexes the operation, so the control transfers.

In [ ]:
AFFINE_OPS = [(1, 1), (1, 3), (3, 0), (5, 0)]


def _pack(feats, n_delay, qpos=None, mark=False, device=DEVICE):
    """Pack a visual trial into one float tensor so every downstream metric works
    unchanged: [features | query one-hot | is_delay | is_query | is_mark]."""
    B, n, _ = feats.shape
    T_ = n + n_delay + 1 + (1 if mark else 0)
    x = torch.zeros(B, T_, IN_DIM, device=device)
    off = 1 if mark else 0
    if mark:
        x[:, 0, D_FEAT + MAX_POS + 2] = 1.0
    x[:, off:off + n, :D_FEAT] = feats
    x[:, off + n:off + n + n_delay, D_FEAT + MAX_POS] = 1.0
    x[:, -1, D_FEAT + MAX_POS + 1] = 1.0
    if qpos is not None:
        x[torch.arange(B, device=device), -1, D_FEAT + qpos] = 1.0
    return x


class Task:
    name, n_out = "base", None

    def __init__(self, cfg=None, split="train"):
        self.cfg = cfg or CFG
        self.split = split
        if IS_VISUAL:
            self.feats = BANK["f_tr"] if split == "train" else BANK["f_te"]
            self.targets = BANK["t_tr"] if split == "train" else BANK["t_te"]

    def _sample(self, B, n, device):
        """Returns (feats_or_none, symbol_labels) for n items."""
        if IS_VISUAL:
            idx = torch.randint(0, self.feats.shape[0], (B, n), device=device)
            return self.feats[idx], self.targets[idx]
        s = torch.randint(0, self.cfg.n_symbols, (B, n), device=device)
        return None, s

    def show(self, load, n=3):
        b = self.generate(n, load, device=DEVICE)
        if IS_VISUAL:
            print(f"  x {tuple(b['x'].shape)} float   y {b['y'][:n].tolist()}")
        else:
            for i in range(n):
                print("  " + " ".join(token_str(t) for t in b["x"][i].cpu())
                      + f"  →  {token_str(int(b['y'][i]) + N_SPECIAL)}")


class SelectiveRecall(Task):
    name = "selective_recall"

    @property
    def n_out(self):
        # honour a non-default alphabet (the swap-error analysis in S6.6 needs one)
        return N_CLASSES if IS_VISUAL else self.cfg.n_symbols

    def generate(self, B, load, extra_delay=0, device=DEVICE, extra_items=0):
        c, n = self.cfg, load + extra_items
        assert n <= c.max_query_pos
        feats, lab = self._sample(B, n, device)
        j = torch.randint(0, load, (B,), device=device)     # trained positions only
        ar = torch.arange(B, device=device)
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=j, device=device)
        else:
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            q = (j + N_SPECIAL + c.n_symbols).unsqueeze(1)
            x = torch.cat([lab + N_SPECIAL, delay, q], 1)
        return {"x": x, "y": lab[ar, j], "key_pos": j, "items": lab}


class DelayedRecall(Task):
    name = "delayed_recall"

    @property
    def n_out(self):
        # honour a non-default alphabet (the swap-error analysis in S6.6 needs one)
        return N_CLASSES if IS_VISUAL else self.cfg.n_symbols

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        feats, lab = self._sample(B, load, device)
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=None, device=device)
        else:
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
            x = torch.cat([lab + N_SPECIAL, delay, cue], 1)
        return {"x": x, "y": lab[:, 0],
                "key_pos": torch.zeros(B, dtype=torch.long, device=device),
                "items": lab}


class Distractor(Task):
    name = "distractor"

    @property
    def n_out(self):
        # honour a non-default alphabet (the swap-error analysis in S6.6 needs one)
        return N_CLASSES if IS_VISUAL else self.cfg.n_symbols

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        feats, lab = self._sample(B, load + 1, device)
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=None, mark=True, device=device)
        else:
            mark = torch.full((B, 1), MARK, device=device, dtype=torch.long)
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
            x = torch.cat([mark, lab + N_SPECIAL, delay, cue], 1)
        return {"x": x, "y": lab[:, 0],
                "key_pos": torch.ones(B, dtype=torch.long, device=device),
                "items": lab}


class AffineTrack(Task):
    """Positive control: non-commutative composition, so a single attention read
    over a bag of values cannot recover the answer."""
    name = "affine_track"
    n_out = 8

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        feats, lab = self._sample(B, load, device)
        ops = lab % len(AFFINE_OPS)
        A = torch.tensor([a for a, _ in AFFINE_OPS], device=device)
        Bv = torch.tensor([b for _, b in AFFINE_OPS], device=device)
        s = torch.zeros(B, dtype=torch.long, device=device)
        for t in range(load):
            o = ops[:, t]
            s = (A[o] * s + Bv[o]) % 8
        nd = c.base_delay + extra_delay
        if IS_VISUAL:
            x = _pack(feats, nd, qpos=None, device=device)
        else:
            delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
            cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
            x = torch.cat([ops + N_SPECIAL, delay, cue], 1)
        return {"x": x, "y": s,
                "key_pos": torch.full((B,), load - 1, dtype=torch.long, device=device),
                "items": ops}


TASKS = {t.name: t for t in [SelectiveRecall, DelayedRecall, Distractor, AffineTrack]}


def make_task(name="selective_recall", cfg=None, split="train"):
    return TASKS[name](cfg, split)

## Models

The additive two-route readout is the whole method:

> **logits = W_h · f(h_T)  +  W_c · g(c)**

`h_T` is the final recurrent state; `c` is a single-query attention read with
the query from `h_T` and keys/values from `H[:, :−1]`. Excluding the final
timestep is necessary — otherwise `v(h_T)` smuggles the recurrent state through
the context path and window ablation measures the wrong thing.

The routes meet only as a sum, so each can be ablated or priced independently.
`attention=False` removes the retrieval route entirely, which is what makes the
§4.1 control an actual control.

In [ ]:
def ablate(v, spec):
    """Resample ablation: replace a trial's activation with another trial's, so the
    readout still sees an in-distribution vector that is uninformative about this
    trial. Zeroing would be off-manifold."""
    if v is None or spec is None:
        return v
    mode, strength = spec if isinstance(spec, (tuple, list)) else (spec, 1.0)
    if strength <= 0:
        return v
    if mode == "noise":
        return v + strength * v.std() * torch.randn_like(v)
    repl = {"shuffle": lambda: v[torch.randperm(v.shape[0], device=v.device)],
            "mean": lambda: v.mean(0, keepdim=True).expand_as(v),
            "zero": lambda: torch.zeros_like(v)}[mode]()
    return (1.0 - strength) * v + strength * repl


class DiagLinearRNN(nn.Module):
    """h_t = a ⊙ h_{t-1} + B x_t — the S4D / LRU core."""

    def __init__(self, d):
        super().__init__()
        self.a_logit = nn.Parameter(torch.linspace(1.0, 4.0, d))
        self.B, self.C, self.gate = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x):
        a = torch.sigmoid(self.a_logit); u = self.B(x)
        h = x.new_zeros(x.shape[0], x.shape[2]); hs = []
        for t in range(x.shape[1]):
            h = a * h + u[:, t]; hs.append(h)
        H = torch.stack(hs, 1)
        return self.C(H) * torch.sigmoid(self.gate(x)), h


class SelectiveSSM(nn.Module):
    """Input-dependent decay and gain — Mamba's selectivity mechanism."""

    def __init__(self, d):
        super().__init__()
        self.a_base = nn.Parameter(torch.linspace(1.0, 4.0, d))
        self.a_sel, self.b_sel = nn.Linear(d, d), nn.Linear(d, d)
        self.B, self.C, self.gate = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x):
        a = torch.sigmoid(self.a_base.view(1, 1, -1) + self.a_sel(x))
        u = torch.sigmoid(self.b_sel(x)) * self.B(x)
        h = x.new_zeros(x.shape[0], x.shape[2]); hs = []
        for t in range(x.shape[1]):
            h = a[:, t] * h + u[:, t]; hs.append(h)
        H = torch.stack(hs, 1)
        return self.C(H) * torch.sigmoid(self.gate(x)), h


EXTRA_CORES = {}          # late-registered cores (see the Mamba cell at the end)


class CausalTransformer(nn.Module):
    """Attention-only sequence core, capacity-matched against the recurrent ones
    for S3.5. It is registered with HAS_ATTN=False because it has no *retrieval
    route in the readout* — self-attention inside the core is not the priced
    pathway. Its role in the paper is the parameter-count reference point."""

    def __init__(self, d, n_layers=2, n_heads=4, max_len=256):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_len, d))
        nn.init.normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=2 * d, dropout=0.0,
            batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, n_layers)

    def forward(self, x):
        T_ = x.shape[1]
        h = x + self.pos[:, :T_]
        mask = torch.triu(torch.ones(T_, T_, device=x.device, dtype=torch.bool), 1)
        H = self.enc(h, mask=mask)
        return H, H[:, -1]


class TwoRoute(nn.Module):
    CORES = {"gru": nn.GRU, "lstm": nn.LSTM, "rnn": nn.RNN}
    K = 10.0

    def __init__(self, cfg=None, core="gru", d_h=None, d_c=None, n_out=None,
                 attention=True, gate=False, exclude_last=True):
        super().__init__()
        cfg = cfg or CFG
        d_h = d_h or cfg.d_model
        d_c = d_c or d_h
        self.cfg, self.d_h, self.d_c = cfg, d_h, d_c
        self.attention, self.use_gate, self.exclude_last = attention, gate, exclude_last
        n_out = n_out or N_CLASSES
        self.emb = nn.Linear(IN_DIM, d_h) if IS_VISUAL else nn.Embedding(vocab_size(cfg), d_h)
        if core in self.CORES:
            self.rnn = self.CORES[core](d_h, d_h, batch_first=True)
        elif core == "linear":
            self.rnn = DiagLinearRNN(d_h)
        elif core == "selective":
            self.rnn = SelectiveSSM(d_h)
        elif core == "transformer":
            self.rnn = CausalTransformer(d_h)
        elif core in EXTRA_CORES:
            self.rnn = EXTRA_CORES[core](d_h)
        else:
            raise ValueError(core)
        self.proj_h = nn.Sequential(nn.Linear(d_h, d_h), nn.GELU())
        self.w_h = nn.Linear(d_h, n_out)
        if attention:
            self.q, self.k, self.v = (nn.Linear(d_h, d_c), nn.Linear(d_h, d_c),
                                      nn.Linear(d_h, d_c))
            self.scale = d_c ** -0.5
            self.proj_c = nn.Sequential(nn.Linear(d_c, d_c), nn.GELU())
            self.w_c = nn.Linear(d_c, n_out, bias=False)
        if gate:
            self.gate_logit = nn.Parameter(torch.tensor(2.0))

    @property
    def z(self):
        return torch.sigmoid(self.gate_logit) if self.use_gate else None

    def attend(self, hq, H, window=None):
        B, T_, _ = H.shape
        sc = (self.q(hq).unsqueeze(1) @ self.k(H).transpose(1, 2)).squeeze(1) * self.scale
        if window is not None:
            idx = torch.arange(T_, device=H.device)
            sc = sc.masked_fill((idx < T_ - window).unsqueeze(0), float("-inf"))
        w = sc.softmax(-1)
        return torch.einsum("bt,btd->bd", w, self.v(H)), w

    def forward(self, x, intervene=None, return_internals=False):
        iv = intervene or {}
        H, _ = self.rnn(self.emb(x))
        h_T = H[:, -1, :]
        h_r = ablate(h_T, iv.get("hidden"))
        if iv.get("erase") is not None:
            P = iv["erase"]; h_r = h_r - (h_r @ P) @ P.T
        lh = self.w_h(self.proj_h(h_r))
        if not self.attention:
            out = {"logits": lh, "parts": {"hidden": lh, "context": torch.zeros_like(lh)}}
            if return_internals:
                out.update({"H": H, "h_T": h_T, "ctx": None, "attn": None})
            return out
        H_ctx = H[:, :-1, :] if self.exclude_last else H
        ctx, aw = self.attend(ablate(h_T, iv.get("query")), H_ctx,
                              window=iv.get("attn_window"))
        ctx = ablate(ctx, iv.get("context"))
        lc = self.w_c(self.proj_c(ctx))
        if self.use_gate:      # mixture gate: normalise BOTH routes, gate the mixture,
            nrm = lambda u: u / (u.norm(dim=-1, keepdim=True) + 1e-6)
            lh, lc = self.K * nrm(lh), self.K * self.z * nrm(lc)   # so no rescaling undoes it
        out = {"logits": lh + lc, "parts": {"hidden": lh, "context": lc}}
        if return_internals:
            out.update({"H": H, "h_T": h_T, "ctx": ctx, "attn": aw})
        return out


def route_params(m):
    """Split the parameters into the two routes plus what they share.

    The mixture gate belongs to the retrieval route: it scales that route's
    contribution, and an LR handicap applied to "attention" should reach it.
    Leaving it out of every group meant it never reached the optimiser at all,
    so z stayed at sigmoid(2.0) for every lambda and the gate cost form
    produced a flat sweep. The assertion in `train` now makes that class of
    omission fail loudly instead of silently.
    """
    rec = list(m.proj_h.parameters()) + list(m.w_h.parameters())
    att = ((list(m.q.parameters()) + list(m.k.parameters()) + list(m.v.parameters())
            + list(m.proj_c.parameters()) + list(m.w_c.parameters()))
           if m.attention else [])
    if getattr(m, "use_gate", False):
        att = att + [m.gate_logit]
    shared = list(m.emb.parameters()) + list(m.rnn.parameters())
    return rec, att, shared


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


CORE_OF = {"rnn": "rnn", "gru": "gru", "lstm": "lstm", "gru_attn": "gru",
           "lstm_attn": "lstm", "linear_attn": "linear", "selective_attn": "selective",
           "transformer": "transformer"}
HAS_ATTN = {"rnn": False, "gru": False, "lstm": False, "gru_attn": True,
            "lstm_attn": True, "linear_attn": True, "selective_attn": True,
            "transformer": False}


CAP_TABLE = []


def match_capacity(cfg=None, verbose=True):
    """Reference is the attention-FREE GRU, so the target reflects a plain
    recurrent model and the attention arms pay for their extra parameters."""
    cfg = cfg or CFG
    target = count_params(TwoRoute(cfg, "gru", d_h=cfg.d_model, attention=False))
    dims, rows = {}, []
    for name in CORE_OF:
        best = min(range(8, 400, 4),
                   key=lambda d: abs(count_params(
                       TwoRoute(cfg, CORE_OF[name], d_h=d,
                                attention=HAS_ATTN[name])) - target))
        dims[name] = best
        rows.append({"model": name, "attention": HAS_ATTN[name], "d": best,
                     "params": count_params(TwoRoute(cfg, CORE_OF[name], d_h=best,
                                                     attention=HAS_ATTN[name]))})
    if verbose:
        print(f"target {target:,} params (attention-free gru @ d={cfg.d_model})")
        print(pd.DataFrame(rows).to_string(index=False))
    CAP_TABLE[:] = rows
    return dims


DIMS = match_capacity()


def build(name="gru_attn", cfg=None, d_h=None, d_c=None, gate=False, n_out=None):
    return TwoRoute(cfg or CFG, CORE_OF[name], d_h=d_h or DIMS[name], d_c=d_c,
                    n_out=n_out, attention=HAS_ATTN[name], gate=gate)

## Training, metrics, crossings

Ablation is by resampling across the batch. Dependencies are normalised by each
model's own headroom above its own chance level, so an 8-way symbolic readout
and a 100-way CIFAR-100 readout give comparable numbers.

`crossing_any` detects a sign change in either direction: on a load axis the
difference goes negative→positive, on a λ axis positive→negative. A
direction-specific version silently returned NaN for every width crossing.

In [ ]:
def train(task, load, lam=0.0, form="l2", seed=0, steps=2500, model=None,
          name="gru_attn", d_h=None, d_c=None, gate=False, schedule="cosine",
          p_drop=0.0, log_every=None, cfg=None, handicap=None, lam_after=None,
          switch=None, ckpt_key=None, ckpt_every=2000):
    """`schedule='constant'` wherever the drift itself is studied — cosine decay
    freezes it partway and confounds the measurement. `handicap` scales a group's
    LEARNING RATE; scaling its gradient is nearly a no-op under Adam.

    `ckpt_key` enables mid-run checkpointing, for the 12k–24k step runs where
    losing the whole run to a disconnect is expensive. Resuming restarts the
    data stream from a fresh RNG state, so a resumed run is not bit-identical to
    an uninterrupted one; it is statistically equivalent because trials are
    generated i.i.d. online. Seeds still fix initialisation exactly.
    """
    cfg = cfg or CFG
    set_seed(seed)
    m = (model or build(name, cfg, d_h, d_c, gate, task.n_out)).to(DEVICE)
    rec, att, shared = route_params(m)
    groups = [{"params": rec, "name": "recurrent"}, {"params": shared, "name": "shared"}]
    if att:
        groups.insert(1, {"params": att, "name": "attention"})
    opt = torch.optim.AdamW(groups, lr=cfg.lr, weight_decay=cfg.weight_decay)
    _grouped = sum(len(g["params"]) for g in groups)
    _total = len([p for p in m.parameters() if p.requires_grad])
    assert _grouped == _total, (
        f"{_total - _grouped} parameter tensor(s) are not in any optimiser group "
        f"and will never be updated. Add them to route_params.")
    trace, start = [], 1

    if ckpt_key is not None:
        ck = ckpt_load(f"run_{ckpt_key}")
        if ck is not None and ck.get("steps") == steps:
            m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            trace, start = ck["trace"], ck["step"] + 1
            if start <= steps:
                print(f"    resumed run {ckpt_key} at step {ck['step']}/{steps}")

    for step in range(start, steps + 1):
        cos = 0.5 * (1 + math.cos(math.pi * step / steps)) if schedule == "cosine" else 1.0
        for g in opt.param_groups:
            f = (handicap["factor"] if handicap and step <= handicap["until"]
                 and g["name"] == handicap["route"] else 1.0)
            g["lr"] = cfg.lr * cos * f
        cur = lam if (switch is None or step <= switch) else lam_after
        m.train()
        b = task.generate(cfg.batch_size, load)
        iv = ({"context": ("zero", 1.0)} if p_drop > 0 and random.random() < p_drop
              else None)
        o = m(b["x"], intervene=iv)
        loss = F.cross_entropy(o["logits"], b["y"])
        if cur and cur > 0 and m.attention:
            lc = o["parts"]["context"]
            loss = loss + cur * ({"l2": lambda: lc.norm(dim=-1).mean(),
                                  "l2sq": lambda: lc.pow(2).sum(-1).mean(),
                                  "gate": lambda: m.z}[form]())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), cfg.grad_clip)
        opt.step()
        if log_every and (step % log_every == 0 or step == 1):
            trace.append({"step": step, **deps(m, task, load, cfg)})
        if ckpt_key is not None and step % ckpt_every == 0 and step < steps:
            ckpt_save(f"run_{ckpt_key}",
                      {"model": m.state_dict(), "opt": opt.state_dict(),
                       "step": step, "steps": steps, "trace": trace})

    if ckpt_key is not None:
        p = ckpt_path(f"run_{ckpt_key}")
        if p.exists():
            p.unlink()          # the finished run lives in the results store now
    return m, pd.DataFrame(trace)


@torch.no_grad()
def accuracy(m, task, load, intervene=None, n_batches=8, cfg=None, **gen):
    cfg = cfg or CFG
    m.eval(); c = t = 0
    for _ in range(n_batches):
        b = task.generate(cfg.batch_size, load, **gen)
        c += (m(b["x"], intervene=intervene)["logits"].argmax(-1) == b["y"]).sum().item()
        t += b["y"].numel()
    m.train(); return c / t


@torch.no_grad()
def deps(m, task, load, cfg=None, n_batches=8, mode="shuffle"):
    cfg = cfg or CFG
    chance = 1.0 / m.w_h.out_features
    acc = accuracy(m, task, load, n_batches=n_batches, cfg=cfg)
    den = max(acc - chance, 1e-6)
    d = lambda a: float(np.clip((acc - a) / den, 0, 1))
    hd = d(accuracy(m, task, load, {"hidden": (mode, 1.0)}, n_batches, cfg))
    if not m.attention:
        return {"acc_clean": acc, "hidden_dep": hd, "attn_dep": 0.0, "context_share": 0.0}
    ad = d(accuracy(m, task, load, {"context": (mode, 1.0)}, n_batches, cfg))
    m.eval(); sh, sc = [], []
    for _ in range(4):
        p = m(task.generate(cfg.batch_size, load)["x"])["parts"]
        sh.append(p["hidden"].std(-1)); sc.append(p["context"].std(-1))
    m.train()
    h, c = torch.cat(sh).mean().item(), torch.cat(sc).mean().item()
    return {"acc_clean": acc, "hidden_dep": hd, "attn_dep": ad,
            "context_share": c / (h + c + 1e-9)}


def anneal(task, load, seed=0, stages=(0.05, 0.1, 0.2, 0.3, 0.5, 0.8),
           target=0.15, steps=2500, name="gru_attn", d_h=None, cfg=None):
    """Learn with retrieval available, then price it out. Training with the route
    masked from the start finds much worse recurrent solutions."""
    m, _ = train(task, load, lam=0.0, seed=seed, steps=steps, name=name, d_h=d_h, cfg=cfg)
    used = None
    for lam in stages:
        m, _ = train(task, load, lam=lam, seed=seed, steps=steps // 2, model=m, cfg=cfg)
        used = lam
        if deps(m, task, load, cfg, n_batches=4)["context_share"] < target:
            break
    return m, used


def crossing_any(x, hidden, attn):
    x = np.asarray(x, float)
    d = np.asarray(attn, float) - np.asarray(hidden, float)
    ok = ~np.isnan(d); x, d = x[ok], d[ok]
    o = np.argsort(x); x, d = x[o], d[o]
    for i in range(len(d) - 1):
        a, b = d[i], d[i + 1]
        if a == 0:
            return float(x[i])
        if a * b < 0:
            f = abs(a) / (abs(a) + abs(b))
            return float(x[i] + f * (x[i + 1] - x[i]))
    return np.nan


def boundary_fit(cross):
    pts = [(math.log2(k), v) for k, v in cross.items()
           if k > 0 and v is not None and not np.isnan(v)]
    if len(pts) < 3:
        return (np.nan, np.nan, np.nan)
    x = np.array([p[0] for p in pts]); y = np.array([p[1] for p in pts])
    b = np.cov(x, y, bias=True)[0, 1] / x.var(); a = y.mean() - b * x.mean()
    ss_t = ((y - y.mean()) ** 2).sum()
    return (a, b, 1 - ((y - (a + b * x)) ** 2).sum() / max(ss_t, 1e-12))


def crossings_by_seed(df, axis="load", keys=None, acc_floor=0.0):
    """Per-seed crossings. Averaging dependencies first and interpolating once
    hides exactly the variance that needs reporting."""
    if acc_floor > 0 and "acc_clean" in df.columns:
        df = df[df.acc_clean >= acc_floor]
    keys = [k for k in (keys or ["lam", "task", "model", "d_h", "d_c", "p_drop", "form"])
            if k in df.columns and k != axis and df[k].nunique() > 1]
    out = []
    for k, g in (df.groupby(keys + ["seed"]) if keys else df.groupby("seed")):
        gg = g.sort_values(axis)
        rec = dict(zip(keys + ["seed"], k if isinstance(k, tuple) else (k,)))
        rec["crossing"] = crossing_any(gg[axis], gg.hidden_dep, gg.attn_dep)
        out.append(rec)
    return pd.DataFrame(out)


def boot_ci(vals, n=10000, alpha=0.05, seed=0):
    v = np.asarray([x for x in vals if x == x], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    mu = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.quantile(mu, alpha/2)), float(np.quantile(mu, 1-alpha/2))


def report_crossings(df, axis="load", label="", keys=None, acc_floor=0.0):
    C = crossings_by_seed(df, axis, keys, acc_floor)
    if not len(C):
        print(f"  {label}: no crossings"); return C
    gk = [c for c in C.columns if c not in ("seed", "crossing")]
    print(f"\n{label}  (axis {axis}" + (f", acc floor {acc_floor}" if acc_floor else "") + ")")
    for k, g in (C.groupby(gk) if gk else [((), C)]):
        mu, lo, hi = boot_ci(g.crossing.values)
        lbl = ", ".join(f"{a}={b}" for a, b in zip(gk, k if isinstance(k, tuple) else (k,))) or "all"
        n_ok = int(g.crossing.notna().sum())
        flag = "  <- off-grid for some seeds" if n_ok < len(g) else ""
        print(f"  {lbl:<34} {mu:7.3f}  95% CI [{lo:.3f}, {hi:.3f}]  n={n_ok}/{len(g)}{flag}")
    if gk and "lam" in gk and axis == "load":
        a, b, r2 = boundary_fit(C.dropna().groupby("lam").crossing.mean().to_dict())
        print(f"  fit: load* = {a:.2f} + {b:.2f}·log2(lambda),  R^2 = {r2:.3f}")
    return C


def sweep(name, conditions, seeds, steps, schedule="cosine", out=None,
          model_fn=None, cfg=None, log_every=None, ckpt=False):
    """Resumable factorial sweep. Every (condition, seed) is hashed and skipped
    if already on disk, so a killed session costs at most one run."""
    st = Store(name)
    units = list(itertools.product(conditions, seeds))
    todo = [(c, sd) for c, sd in units
            if not st.has({**c, "seed": sd, "steps": steps, "schedule": schedule,
                           "block": name, "cfg": cfg_tag(cfg)})]
    print(f"  [{name}] {len(units)} units, {len(units)-len(todo)} cached, {len(todo)} to run")
    t0, done = time.time(), 0
    try:
        for c, sd in todo:
            spec = {**c, "seed": sd, "steps": steps, "schedule": schedule,
                    "block": name, "cfg": cfg_tag(cfg)}
            t1 = time.time()
            tk = make_task(c.get("task", "selective_recall"), cfg)
            m, tr = train(tk, c["load"], lam=c.get("lam", 0.0), form=c.get("form", "l2"),
                          seed=sd, steps=steps, model=model_fn(c) if model_fn else None,
                          name=c.get("model", "gru_attn"), d_h=c.get("d_h"),
                          d_c=c.get("d_c"), gate=c.get("gate", False),
                          schedule=schedule, p_drop=c.get("p_drop", 0.0),
                          log_every=log_every, cfg=cfg,
                          ckpt_key=(keyof(spec) if ckpt else None))
            payload = dict(deps(m, tk, c["load"], cfg))
            if len(tr):
                payload["trace"] = tr.to_dict("records")
            payload["_elapsed"] = time.time() - t1
            st.put(spec, payload)
            done += 1
            if done % 15 == 0 or done == len(todo):
                el = (time.time() - t0) / 60
                print(f"  [{name}] {done}/{len(todo)}  {el:.1f}m, ~{el/done*(len(todo)-done):.1f}m left")
    except KeyboardInterrupt:
        print(f"  [{name}] interrupted at {done}/{len(todo)} — finished runs are saved, "
              f"rerun this cell to continue")
    df = st.frame()
    if out and len(df):
        df.to_csv(out, index=False)
        tf = st.traces()
        if len(tf):
            tf.to_csv(out.replace(".csv", "_traces.csv"), index=False)
    return df


if RUN["sanity"]:
    for nm in TASKS:
        print(f"\n{nm} (load 4, n_out={make_task(nm).n_out})")
        make_task(nm).show(4, 2)
    CAP = pd.DataFrame(CAP_TABLE)
    CAP.to_csv(csv("p0_capacity"), index=False)
    print("\ncapacity matching (S3.5):")
    print(CAP.to_string(index=False))


In [ ]:
# CIFAR-10 as a third stimulus type. Same task, same readout, same ablations,
# same metrics — only the encoder differs, so a difference in result cannot be a
# difference in measurement.
#
# MNIST taught the protocol: conditions copied from the symbolic setup did not
# fire, because a frozen CNN feature is a heavier burden on the recurrent state
# and the boundary sits lower (intercept 14.01 -> 7.02, slope 1.46 -> 0.62). So
# sufficiency is measured first and the conditions are placed from this
# modality's own crossing. CIFAR-10 features carry more variance per item than
# MNIST's, so expect a narrower viable range still — possibly load 2-3 only.

assert MODALITY == "cifar10", "set MODALITY = 'cifar10' and re-run the setup cell"

for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"modality {MODALITY}, {N_CLASSES} classes, chance {1/N_CLASSES:.3f}")
print(f"results -> {RESDIR}\n")


# ---------------------------------------------------------------------------
# Seeds
# ---------------------------------------------------------------------------

# Keep the expensive exploratory/boundary experiments on the existing seed set.
# Use 8 seeds only for the headline CIFAR inversion + late-switch experiments.
SEEDS_CIFAR = list(range(8))

print(f"auxiliary/boundary seeds: {SEEDS_AUX}")
print(f"headline CIFAR seeds:     {SEEDS_CIFAR}")


def plateau_step(curve, frac=0.99, tail=5):
    """
    First step at which accuracy reaches `frac` of its own plateau.

    A fixed threshold does not transfer across stimulus types: the symbolic
    ceiling is 1.000, MNIST's is ~0.99, and CIFAR-10's is unknown in advance.
    Defining convergence relative to what a condition actually achieves keeps
    the 'behaviour has stopped changing' criterion comparable across modalities.
    """
    plat = curve.iloc[-tail:].mean()
    hit = curve.index[curve >= frac * plat]
    return (int(hit.min()) if len(hit) else None), float(plat)


# ---------------------------------------------------------------------------
# §4.1 — Is retrieval used when both algorithms are free?
# ---------------------------------------------------------------------------

conds = [
    {"task": "selective_recall", "model": m, "load": L}
    for m in ["gru_attn", "gru"]
    for L in [2, 3, 4, 8, 12]
]

FR = sweep(
    "free",
    conds,
    SEEDS_AUX,
    STEPS,
    out=csv("p1_free"),
)

print("hybrid at lambda=0 — hidden_dep (top) and accuracy (bottom):")

print(
    FR[FR.model == "gru_attn"]
    .pivot_table(index="model", columns="load", values="hidden_dep")
    .round(3)
    .to_string()
)

print(
    FR[FR.model == "gru_attn"]
    .pivot_table(index="model", columns="load", values="acc_clean")
    .round(3)
    .to_string()
)


# ---------------------------------------------------------------------------
# Is the recurrent alternative reachable, at 3x budget?
# ---------------------------------------------------------------------------

SUF_STEPS = 3 * STEPS

conds = [
    {"task": "selective_recall", "model": "gru", "load": L}
    for L in [2, 3, 4, 6, 8]
]

FL = sweep(
    "free_long",
    conds,
    SEEDS_AUX,
    SUF_STEPS,
    out=csv("p1_free_long"),
)

suff = FL.groupby("load").acc_clean.agg(["mean", "std"])

print(f"\nattention-free accuracy at {SUF_STEPS} steps:")
print(suff.round(3).to_string())

SUF_GATE = 0.95

VIABLE = [
    int(L)
    for L, r in suff.iterrows()
    if r["mean"] >= SUF_GATE
]

print(f"\n  loads where recurrence alone reaches {SUF_GATE}: {VIABLE or 'none'}")

if not VIABLE:
    print(
        "  The inversion cannot occur where there is no viable recurrent"
    )
    print(
        "  alternative — the price has nowhere to send the computation. That is"
    )
    print(
        "  a reportable bound on the phenomenon, not a failed experiment: the"
    )
    print(
        "  effect requires a recurrent solution that is both reachable and"
    )
    print(
        "  sufficient, and the load at which that fails is stimulus-dependent"
    )
    print(
        "  (abstract 8, MNIST 4, here whatever this table says)."
    )


# ---------------------------------------------------------------------------
# §4.2 — Boundary
# ---------------------------------------------------------------------------

conds = [
    {"lam": l, "load": L}
    for l in [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
    for L in [2, 3, 4, 6, 8, 12]
]

BND = sweep(
    "boundary",
    conds,
    SEEDS_AUX,
    STEPS,
    out=csv("p2_boundary"),
)

print("\ncontext_share (rows lambda, cols load):")
print(
    BND.pivot_table(
        index="lam",
        columns="load",
        values="context_share",
    )
    .round(3)
    .to_string()
)

print("\naccuracy:")
print(
    BND.pivot_table(
        index="lam",
        columns="load",
        values="acc_clean",
    )
    .round(3)
    .to_string()
)

print(f"\nminimum accuracy over {len(BND)} runs: {BND.acc_clean.min():.4f}")

CB = report_crossings(
    BND,
    "load",
    f"§4.2 boundary [{MODALITY}]",
    acc_floor=ACC_FLOOR,
)

CB.to_csv(
    csv("p2_boundary_crossings"),
    index=False,
)


# ---------------------------------------------------------------------------
# Place the conditions from the measured boundary
# ---------------------------------------------------------------------------

Cx = (
    CB.dropna(subset=["crossing"])
    if "crossing" in CB
    else pd.DataFrame()
)

have_fit = len(Cx) and Cx.lam.nunique() >= 3

if have_fit:

    a_fit, b_fit, _ = boundary_fit(
        Cx.groupby("lam").crossing.mean().to_dict()
    )

    lam_star = lambda L: 2 ** ((L - a_fit) / b_fit)

    print(
        f"\nmeasured boundary: "
        f"load* = {a_fit:.2f} + {b_fit:.2f}·log2(lambda)"
    )

    for L in [2, 3, 4, 6, 8]:
        print(
            f"  load {L:>2}: "
            f"crossing price lambda* = {lam_star(L):.4f}"
        )

else:

    a_fit = b_fit = np.nan
    lam_star = lambda L: 0.05

    print(
        "\n  !! too few crossings to fit a boundary; "
        "falling back to lambda=0.05"
    )


L_inv = max(VIABLE) if VIABLE else 2

LAMS = [
    round(float(lam_star(L_inv)) * k, 4)
    for k in (1.5, 3.0)
]

L_ctl = min(
    [L for L in [3, 4, 6, 8, 12] if L > L_inv]
    or [L_inv * 2]
)

print(f"\n  inversion: load {L_inv} at lambda {LAMS}")
print(
    f"  control:   load {L_ctl} "
    f"at lambda {LAMS[0]}, past its crossing"
)
print(
    "  The control is a prediction: the fit says this load should not invert"
)
print(
    "  at this price. On MNIST the same procedure predicted load 8 correctly."
)


# ---------------------------------------------------------------------------
# §5.1 — Headline inversion
#
# IMPORTANT:
# This block now uses 8 CIFAR seeds.
# Boundary fitting above remains on SEEDS_AUX.
# ---------------------------------------------------------------------------

conds = (
    [{"lam": l, "load": L_inv} for l in LAMS]
    + [{"lam": LAMS[0], "load": L_ctl}]
)

INV = sweep(
    "inversion",
    conds,
    SEEDS_CIFAR,       # <-- 8 seeds here
    LONG,
    schedule="constant",
    out=csv("p8_inversion"),
    log_every=1000,
    ckpt=True,
)

T = pd.read_csv(
    csv("p8_inversion").replace(
        ".csv",
        "_traces.csv",
    )
)

valid_conditions = {
    (c["lam"], c["load"])
    for c in conds
}

T = T[
    [
        (r.lam, r.load) in valid_conditions
        for r in T.itertuples()
    ]
]


print(
    f"\n§5.1 [{MODALITY}] "
    f"route dependence from convergence to the end:"
)

rows = []

for (lam, L), g in T.groupby(["lam", "load"]):

    curve = g.groupby("step").acc_clean.mean()

    s0, plat = plateau_step(curve)

    if s0 is None:
        print(
            f"  lam={lam:<7} "
            f"L={int(L):>2}  "
            "no plateau found"
        )
        continue

    s1 = int(g.step.max())

    a_ = g[g.step == s0]
    b_ = g[g.step == s1]

    win = g[g.step >= s0]

    # Existing paper-compatible count:
    # a seed is counted if it crosses at any post-convergence checkpoint.
    n_inv = sum(
        int(
            (
                gs.sort_values("step").hidden_dep
                - gs.sort_values("step").attn_dep
                > 0
            ).any()
        )
        for _, gs in g[g.step >= s0].groupby("seed")
    )

    inv = (
        a_.attn_dep.mean() > a_.hidden_dep.mean()
        and b_.hidden_dep.mean() > b_.attn_dep.mean()
    )

    rows.append(
        {
            "modality": TAG,
            "lam": lam,
            "load": L,
            "converged_step": s0,
            "window": s1 - s0,
            "plateau_acc": plat,
            "hidden_0": a_.hidden_dep.mean(),
            "hidden_1": b_.hidden_dep.mean(),
            "attn_0": a_.attn_dep.mean(),
            "attn_1": b_.attn_dep.mean(),
            "acc_mean": win.acc_clean.mean(),
            "acc_sd": win.acc_clean.std(),
            "seeds_inverted": n_inv,
            "n_seeds": g.seed.nunique(),
            "inverted": inv,
            "is_control": L == L_ctl,
        }
    )

    tag = (
        "  CONTROL"
        if L == L_ctl
        else ("   INVERTS" if inv else "")
    )

    print(
        f"  lam={lam:<7} "
        f"L={int(L):>2}  "
        f"plateau {plat:.3f}, "
        f"converged {s0:>5}, "
        f"+{s1-s0:>5} steps  "
        f"hidden "
        f"{a_.hidden_dep.mean():.3f}->"
        f"{b_.hidden_dep.mean():.3f}  "
        f"attn "
        f"{a_.attn_dep.mean():.3f}->"
        f"{b_.attn_dep.mean():.3f}  "
        f"acc {win.acc_clean.mean():.4f}"
        f"+/-{win.acc_clean.std():.4f}  "
        f"{n_inv}/{g.seed.nunique()} seeds"
        f"{tag}"
    )


if rows:

    IVc = pd.DataFrame(rows)

    IVc.to_csv(
        csv("p8_inversion_summary"),
        index=False,
    )

    thin = IVc[
        IVc.plateau_acc < 0.90
    ]

    if len(thin):

        print(
            "\n  !! conditions whose plateau is below 0.90 — "
            "the model is not"
        )

        print(
            "     solving the task there, so a route change cannot be read as"
        )

        print(
            "     'same behaviour, different algorithm':"
        )

        print(
            thin[
                [
                    "lam",
                    "load",
                    "plateau_acc",
                ]
            ]
            .round(3)
            .to_string(index=False)
        )


# ---------------------------------------------------------------------------
# §5.2 — Four-arm late-switch experiment
#
# IMPORTANT:
# This block also now uses 8 CIFAR seeds.
# ---------------------------------------------------------------------------

LAM_S = LAMS[-1]
SW = 2000
ST = S(12000, 4000, 400)

ARMS = {
    "maintained": (LAM_S, LAM_S),
    "withdrawn": (LAM_S, 0.0),
    "never": (0.0, 0.0),
    "late": (0.0, LAM_S),
}

st = Store("switch")

for (arm, (lb, la)), sd in itertools.product(
    ARMS.items(),
    SEEDS_CIFAR,       # <-- 8 seeds here
):

    spec = {
        "block": "switch",
        "arm": arm,
        "load": L_inv,
        "seed": sd,
        "lam": LAM_S,
        "switch": SW,
        "steps": ST,
    }

    def _go(
        arm=arm,
        lb=lb,
        la=la,
        sd=sd,
        spec=spec,
    ):

        task = make_task(
            "selective_recall"
        )

        m, tr = train(
            task,
            L_inv,
            lam=lb,
            lam_after=la,
            switch=SW,
            seed=sd,
            steps=ST,
            schedule="constant",
            log_every=500,
            ckpt_key=keyof(spec),
        )

        at = (
            tr[
                tr.step <= SW
            ]
            .context_share
            .iloc[-1]
        )

        return {
            "share_at_switch": float(at),
            **deps(
                m,
                task,
                L_inv,
            ),
            "trace": tr.to_dict(
                "records"
            ),
        }

    cached(
        st,
        spec,
        _go,
        label=(
            f"{arm:<11} "
            f"load {L_inv} "
            f"seed {sd}"
        ),
    )


SWI = st.frame()

SWI.to_csv(
    csv("p9_switch"),
    index=False,
)

st.traces().to_csv(
    csv("p9_switch_traces"),
    index=False,
)


print(
    f"\n§5.2 [{MODALITY}] "
    f"load {L_inv}, "
    f"lambda {LAM_S}:"
)

for arm in [
    "never",
    "maintained",
    "late",
    "withdrawn",
]:

    g = SWI[
        SWI.arm == arm
    ]

    if len(g):

        print(
            f"  {arm:<11} "
            f"{g.share_at_switch.mean():.3f} -> "
            f"{g.context_share.mean():.3f}"
            f"+/-{g.context_share.std():.3f}"
            f"   hidden_dep "
            f"{g.hidden_dep.mean():.3f}"
            f"   acc "
            f"{g.acc_clean.mean():.4f}"
        )


lt = SWI[
    SWI.arm == "late"
]

mt = SWI[
    SWI.arm == "maintained"
]

if len(lt) and len(mt):

    gap = abs(
        lt.context_share.mean()
        - mt.context_share.mean()
    )

    sds = (
        lt.context_share.std()
        + mt.context_share.std()
    )

    verdict = (
        "indistinguishable"
        if gap <= sds
        else "DIFFERENT"
    )

    print(
        f"\n  late and maintained differ by "
        f"{gap:.3f}, summed seed SDs "
        f"{sds:.3f} -> {verdict}"
    )

    print(
        "  They start from opposite points, so converging endpoints mean the"
    )

    print(
        "  price reorganises an established retrieval solution, not only a"
    )

    print(
        "  forming one."
    )


# ---------------------------------------------------------------------------
# All modalities side by side
# ---------------------------------------------------------------------------

NICE = {
    "abstract": "symbols",
    "mnist": "MNIST",
    "cifar10": "CIFAR-10",
    "cifar100": "CIFAR-100",
}

rows = []

for tag in [
    "abstract",
    "mnist",
    "cifar10",
    "cifar100",
]:

    p = str(
        RESDIR
        / f"p2_boundary__{tag}.csv"
    )

    if not os.path.exists(p):
        continue

    d = pd.read_csv(p)

    C = crossings_by_seed(
        d,
        "load",
        ["lam"],
        acc_floor=ACC_FLOOR,
    ).dropna()

    if C.lam.nunique() < 3:
        continue

    per = [
        boundary_fit(
            g.set_index("lam")
            .crossing
            .to_dict()
        )
        for _, g in C.groupby("seed")
    ]

    per = [
        (a, b, r)
        for a, b, r in per
        if b == b
    ]

    sl = [
        b
        for _, b, _ in per
    ]

    mb, lb, hb = (
        boot_ci(sl)
        if len(sl) > 1
        else (
            sl[0],
            np.nan,
            np.nan,
        )
    )

    sp = str(
        RESDIR
        / f"p1_free_long__{tag}.csv"
    )

    vmax = np.nan

    if os.path.exists(sp):

        s_ = (
            pd.read_csv(sp)
            .groupby("load")
            .acc_clean
            .mean()
        )

        ok = [
            L
            for L, v in s_.items()
            if v >= SUF_GATE
        ]

        vmax = (
            max(ok)
            if ok
            else np.nan
        )

    rows.append(
        {
            "stimuli": NICE.get(
                tag,
                tag,
            ),
            "seeds": d.seed.nunique(),
            "intercept": np.mean(
                [
                    a
                    for a, _, _ in per
                ]
            ),
            "slope": mb,
            "slope lo": lb,
            "slope hi": hb,
            "R2": np.mean(
                [
                    r
                    for _, _, r in per
                ]
            ),
            "viable to load": vmax,
        }
    )


if rows:

    XM = pd.DataFrame(rows)

    XM.to_csv(
        str(
            RESDIR
            / "x_modality_boundary.csv"
        ),
        index=False,
    )

    print(
        "\nthe boundary across stimulus types:"
    )

    print(
        XM.round(3)
        .to_string(index=False)
    )

    print(
        "\n  What replicates is the form — monotone, log-linear, at ceiling"
    )

    print(
        "  accuracy. The coefficients fall as the stimulus gets richer, and so"
    )

    print(
        "  does the load at which the recurrent alternative stops being viable."
    )

    print(
        "  Both track the same thing: how much the recurrent state has to hold."
    )


# ---------------------------------------------------------------------------
# Backup
# ---------------------------------------------------------------------------
